# Homework 6: Decision Trees and Ensemble Learning

This notebook works through the 2026 Machine Learning Zoomcamp tree homework using the pinned car fuel-efficiency dataset. We predict **fuel efficiency (MPG)** and compare a shallow decision tree, random forests, and XGBoost.

Each section explains the modeling choice, runs the requested experiment, and prints the result so you can check the answer. The train/validation/test split and random seeds follow the assignment exactly.

## 1. Setup and data

We fill missing values with zero as instructed. We first hold out 20% as the test set, then split the remaining data into 75% training and 25% validation. That gives 60% / 20% / 20% overall. Categorical columns are expanded with `DictVectorizer`; numeric columns stay numeric.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

DATA_PATH = Path('car_fuel_efficiency_2026.csv')
TARGET = 'fuel_efficiency_mpg'
df = pd.read_csv(DATA_PATH).fillna(0)
print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Missing values after filling: {int(df.isna().sum().sum())}')
display(df.head())

Dataset shape: 10,000 rows × 11 columns
Missing values after filling: 0


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,0.0,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,0.0,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [2]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

y_train = df_train[TARGET].to_numpy()
y_val = df_val[TARGET].to_numpy()
y_test = df_test[TARGET].to_numpy()

def make_features(train_df, other_df):
    """Fit one-hot encoding on train only, then transform the other split."""
    train_dicts = train_df.drop(columns=TARGET).to_dict(orient='records')
    other_dicts = other_df.drop(columns=TARGET).to_dict(orient='records')
    vectorizer = DictVectorizer(sparse=True)
    X_train = vectorizer.fit_transform(train_dicts)
    X_other = vectorizer.transform(other_dicts)
    return X_train, X_other, vectorizer

X_train, X_val, dv = make_features(df_train, df_val)
# Also build a test matrix with the already-fitted training vocabulary.
X_test = dv.transform(df_test.drop(columns=TARGET).to_dict(orient='records'))
print(f'Train: {X_train.shape}; validation: {X_val.shape}; test: {X_test.shape}')
print('Example encoded feature names:', dv.get_feature_names_out()[:12])

Train: (6000, 16); validation: (2000, 16); test: (2000, 16)
Example encoded feature names: ['acceleration' 'drivetrain=All-wheel drive'
 'drivetrain=Front-wheel drive' 'drivetrain=Rear-wheel drive'
 'engine_displacement' 'fuel_type=Diesel' 'fuel_type=Gasoline'
 'fuel_type=Hybrid' 'horsepower' 'model_year' 'num_cylinders' 'num_doors']


## 2. Question 1 — decision tree split

A depth-one decision tree makes one split. The transformed feature name tells us which column was selected; if the tree chooses an encoded category, we would map it back to its original column.

In [3]:
tree = DecisionTreeRegressor(max_depth=1)
tree.fit(X_train, y_train)
split_feature = dv.get_feature_names_out()[tree.tree_.feature[0]]
original_feature = split_feature.split('=', 1)[0]
print(f'Encoded split feature: {split_feature}')
print(f'Original feature: {original_feature}')

Encoded split feature: model_year
Original feature: model_year


## 3. Question 2 — random forest with 10 trees

RMSE is the square root of mean squared error, so it is expressed in the target's MPG units. We fit the requested forest and evaluate it on validation data.

In [4]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

rf_10 = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf_10.fit(X_train, y_train)
rmse_10 = rmse(y_val, rf_10.predict(X_val))
print(f'Validation RMSE: {rmse_10:.6f} MPG (≈ {rmse_10:.3f})')

Validation RMSE: 1.833220 MPG (≈ 1.833)


## 4. Question 3 — number of trees

We hold the data split and seed fixed while varying `n_estimators`. For this question the model uses the estimator's default depth. The lowest RMSE, rounded to three decimals, determines the answer.

In [5]:
n_estimators_values = [10, 50, 100, 150]
question3 = []
for n in n_estimators_values:
    model = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    model.fit(X_train, y_train)
    score = rmse(y_val, model.predict(X_val))
    question3.append({'n_estimators': n, 'validation_rmse': score, 'rmse_3dp': round(score, 3)})
q3_results = pd.DataFrame(question3)
display(q3_results)
best_n = q3_results.loc[q3_results['validation_rmse'].idxmin(), 'n_estimators']
print(f'Lowest validation RMSE: n_estimators={int(best_n)}')

,n_estimators,validation_rmse,rmse_3dp
0,10,1.833220,1.833
1,50,1.773988,1.774
2,100,1.769786,1.770
3,150,1.769198,1.769


Lowest validation RMSE: n_estimators=150


## 5. Question 4 — choose maximum depth

For each requested depth, we evaluate four forest sizes and average their validation RMSEs. The depth with the lowest mean is the answer. This comparison treats each tree count equally.

In [6]:
depths = [10, 15, 20, 25]
q4_rows = []
for depth in depths:
    scores = []
    for n in n_estimators_values:
        model = RandomForestRegressor(
            n_estimators=n, max_depth=depth, random_state=1, n_jobs=-1
        )
        model.fit(X_train, y_train)
        scores.append(rmse(y_val, model.predict(X_val)))
    q4_rows.append({
        'max_depth': depth,
        **{f'rmse_{n}_trees': score for n, score in zip(n_estimators_values, scores)},
        'mean_rmse': float(np.mean(scores)),
    })
q4_results = pd.DataFrame(q4_rows)
display(q4_results)
best_depth = q4_results.loc[q4_results['mean_rmse'].idxmin(), 'max_depth']
print(f'Lowest mean validation RMSE: max_depth={int(best_depth)}')

,max_depth,rmse_10_trees,rmse_50_trees,rmse_100_trees,rmse_150_trees,mean_rmse
0,10,1.782348,1.747520,1.743986,1.743716,1.754392
1,15,1.831390,1.770883,1.767872,1.766752,1.784224
2,20,1.837314,1.772377,1.768918,1.768476,1.786771
3,25,1.836852,1.772123,1.768320,1.767927,1.786305


Lowest mean validation RMSE: max_depth=10


## 6. Question 5 — feature importance

Random forests report impurity-based importance for every encoded feature. Because one original categorical field can produce multiple encoded columns, we aggregate those values back to their source columns before comparing the four features listed in the homework.

In [7]:
importance_model = RandomForestRegressor(
    n_estimators=10, max_depth=20, random_state=1, n_jobs=-1
)
importance_model.fit(X_train, y_train)
feature_names = dv.get_feature_names_out()
importance_by_column = {}
for name, value in zip(feature_names, importance_model.feature_importances_):
    original = name.split('=', 1)[0]
    importance_by_column[original] = importance_by_column.get(original, 0.0) + float(value)
importance_table = (
    pd.DataFrame({'feature': list(importance_by_column), 'importance': list(importance_by_column.values())})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)
display(importance_table.head(12))
choices = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
most_important = max(choices, key=lambda name: importance_by_column.get(name, 0.0))
print(f'Most important among the four choices: {most_important}')

,feature,importance
0,model_year,0.296107
1,vehicle_weight,0.203313
2,fuel_type,0.193555
3,horsepower,0.078271
4,engine_displacement,0.061281
5,acceleration,0.054998
6,origin,0.052103
7,drivetrain,0.033651
8,num_doors,0.019401
9,num_cylinders,0.007318


Most important among the four choices: vehicle_weight


## 7. Question 6 — XGBoost learning rate

We train two gradient-boosted tree models for 100 rounds, changing only `eta` (the learning rate). Both models use the same training/validation matrices, seed, and RMSE evaluation metric. The lower validation RMSE wins.

In [8]:
import xgboost as xgb

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_names.tolist())
dval = xgb.DMatrix(X_val, label=y_val, feature_names=feature_names.tolist())
watchlist = [(dtrain, 'train'), (dval, 'validation')]
xgb_scores = {}
xgb_models = {}
for eta in [0.3, 0.1]:
    params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 1,
        'eval_metric': 'rmse',
    }
    model = xgb.train(params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
    xgb_models[eta] = model
    xgb_scores[eta] = rmse(y_val, model.predict(dval))
xgb_results = pd.DataFrame([
    {'eta': eta, 'validation_rmse': score}
    for eta, score in xgb_scores.items()
]).sort_values('eta', ascending=False)
display(xgb_results)
best_eta = min(xgb_scores, key=xgb_scores.get)
print(f'Lower validation RMSE: eta={best_eta}')

,eta,validation_rmse
0,0.3,1.826458
1,0.1,1.724810


Lower validation RMSE: eta=0.1


## 8. Answers at a glance

The table below collects the selected homework options from the calculations above. The held-out test data was not used for model selection; it remains available for a final evaluation if you want to extend this exercise.

In [9]:
answers = pd.DataFrame([
    {'Question': '1 — depth-1 split feature', 'Answer': original_feature},
    {'Question': '2 — 10-tree forest validation RMSE', 'Answer': f'{rmse_10:.3f} MPG'},
    {'Question': '3 — best number of trees', 'Answer': int(best_n)},
    {'Question': '4 — best max_depth by mean RMSE', 'Answer': int(best_depth)},
    {'Question': '5 — most important listed feature', 'Answer': most_important},
    {'Question': '6 — best XGBoost eta', 'Answer': best_eta},
])
display(answers)

,Question,Answer
0,1 — depth-1 split feature,model_year
1,2 — 10-tree forest validation RMSE,1.833 MPG
2,3 — best number of trees,150
3,4 — best max_depth by mean RMSE,10
4,5 — most important listed feature,vehicle_weight
5,6 — best XGBoost eta,0.1


### Notes

- The assignment pins the data release and split so the results are comparable to the multiple-choice options.
- Floating point scores can vary slightly across library versions; the choices should still identify the same answer.
- To rerun this notebook locally, activate `.venv` and start Jupyter from this folder. The CSV is stored beside the notebook.